Algo muito positivo sobre o RI do Itaú é que ele é muito similar ao extrator do banco inter

Importante: como classificar a eficiência da sua solução:
1. ela funciona em outros RIs com o mesmo formato?
2. ela é de fácil manutenção?

Encontrado problema: IFRS tem informações, mas sempre vem atrasado. BrGaap vem mais cedo, mas é sempre em PDF no formato que não dá pra copiar. 

formato com estágios: IFRS ( verificar qual página bate )
formato com todo o resto: BrGaap

### Extração de dados pelo BrGaap

1° tentativa é com um print

In [6]:
import pytesseract  # para OCR
from PIL import Image  # para melhorar qualidade da imagem
from pdf2image import convert_from_path  # para pegar uma página específica do PDF
import cv2
import numpy as np

In [2]:
# paths para o programa
path = r"C:\Program Files\Tesseract-OCR" 
path_poppler = r"C:\Users\brono\Downloads\Projetos, códigos, etc\Aplicativos\Poppler\poppler-22.04.0\Library\bin"

In [7]:
# extrai as imagens

RI = "BrGaap - Demonstrações Contábeis 3T25.pdf"

imagens = convert_from_path(
    RI,
    dpi=300,  # obrigatório para aumentar qualidade
    # intervalo de páginas a ser extraído
    first_page=32,
    last_page=33
)

# Adaptando para ficar em escala de cinza

# PIL → NumPy
img_np = np.array(imagens[0])  # imagens[0] é uma PIL.Image

# 1. escala de cinza
cinza = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

# 2. inverter (texto branco → preto)
inverter = cv2.bitwise_not(cinza)

# 3. binarização
th = cv2.adaptiveThreshold(
    inverter,
    255, # inverte de branco para preto
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    31,
    5
)


In [8]:
# Salva na memória

imagem = imagens[0]  # PIL.Image

# OCR direto na imagem em memória
texto = pytesseract.image_to_string(
    imagem,
    lang="por"
)

print(texto)

problema: não está indo a imagem. Faça edições, adapte para uma página, etc

2° tentativa é por página

### Extração de dados pelo IFRS ( aqui tem o perdas esperadas )

In [1]:
# Ingerir PDF
import pdfplumber
import pandas as pd
import re
import numpy as np
import re

In [3]:
with pdfplumber.open("Demonstrações contábeis consolidadas Caixa 2T25.pdf") as pdf:
    page = pdf.pages[51]   # página 53 (índice começa em 0)
    text = page.extract_text()


# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("(a) Movimentação da provisão para perdas esperadas")
end = text.find("(b) Movimentação da provisão para perdas esperadas", start)

trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()]

(a) Movimentação da provisão para perdas esperadas
Saldo em Constituição/ Transferência do/ para Transferência do/ para Saldo em
Estágio 1
31/12/2024 (reversão) estágio 2 estágio 3 30/06/2025
Empréstimos e direitos creditórios descontados (8.366.631) 1.183.160 1.939.179 1.280.197 (3.964.095)
Financiamentos (263.576) 13.504 59.675 48.588 (141.809)
Financiamentos rurais e agroindustriais (1.414.489) 340.925 256.982 214.370 (602.212)
Financiamentos imobiliários (6.702.364) 1.918.808 (307.730) 286.260 (4.805.026)
Financiamentos de infraestrutura (625.891) (30.945) (408) 23 (657.221)
Outros ativos (849.238) 64.789 252.725 77.253 (454.471)
Total (18.222.189) 3.490.241 2.200.423 1.906.691 (10.624.834)
Saldo em Constituição/ Transferência do/ para Transferência do/para Saldo em
Estágio 1
31/12/2023 (reversão) estágio 2 estágio 3 31/12/2024
Empréstimos e direitos creditórios descontados (7.890.186) (560.072) (153.942) 237.569 (8.366.631)
Financiamentos (113.304) (154.655) (2.933) 7.316 (263.576

In [ ]:
# Qual será o formato da tabela, neste caso?